# ワークフローの基本機能

このノートブックでは、フレキシブルワークフローの基本構成を学びます。

フレキシブルワークフローでは、次のように、役割の異なるノードを組み合わせたワークフローを構成します。

- タスクノード：ユーザーと会話しながら必要な情報を聞き出して、情報が揃ったら次のノードに進む
- エージェントノード：ユーザーとの会話は行わず、インストラクションで指示された処理を1回だけ行う
- 関数ノード：通常の関数で直前のノードの出力結果を受け取り、必要な処理を行う
- 分岐処理：特定のノードの処理結果に応じて、次に進むノードを決定する

## 事前準備

**[WBF-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[WBF-02]**

インストールされたパッケージのバージョンを確認します。

In [3]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[WBF-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[WBF-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

**[WBF-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [102]:
import os
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk import Event, Workflow
from google.adk.agents.llm_agent import LlmAgent
from google.adk.workflow import DEFAULT_ROUTE

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[WBF-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

ワークフローの進捗にあわせて結果を表示する `async_output` オプションを追加しています。

In [103]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message, async_output=False):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
                    if async_output:
                        display(Markdown(f'**[{event["author"]}]**'))
                        display(Markdown(response))
        if not async_output:
            return '\n'.join(result)

## タスクノードとエージェントノードの定義

**[WBF-07]**

ユーザーの情報（名前と趣味）を収集するタスクノードを定義します。

In [104]:
class UserInformation(BaseModel):
    """ユーザーの情報"""
    name: str = Field(description='ユーザーの名前')
    hobby: str = Field(description='ユーザーの趣味')

instruction = '''
# タスク
1. ユーザーの名前と趣味を質問します。
2. 得られた情報を UserInformation にセットします。
3. UserInformation の情報が揃ったらタスクを終了します。

# 条件
- フレンドリーに会話してください。
- できるだけ名前と趣味をまとめて聞いてください。
'''

user_information_task = LlmAgent(
    name='user_information_task',
    model='gemini-3.5-flash-lite',
    mode='task',
    output_schema=UserInformation,
    description='ユーザーの情報を集めるタスク',
    instruction=instruction,
)

**[WBF-08]**

タスクノードの実行結果をメッセージとして出力する関数ノードを用意します。

In [105]:
async def output_user_information(node_input: UserInformation):
    message = f'''
```
ユーザー情報を確認しました。
・ 名前: {node_input.name}
・ 趣味: {node_input.hobby}
```
'''
    return Event(
        node_output=node_input,
        message=message,
    )

**[WBF-09]**

ユーザーへのあいさつのメッセージを出力するエージェントノードを定義します。

In [106]:
instruction = '''
ユーザー情報に基づいて、おすすめの旅行先（海外もしくは国内）を提案します。
簡単な一文で出力すること。
'''

travel_guide_agent = LlmAgent(
    name='travel_guide_agent',
    model='gemini-3.5-flash-lite',
    description='おすすめの旅行先を提案するエージェント',
    instruction=instruction,
    input_schema=UserInformation,
)

## ワークフローグラフの定義とワークフローの実行

**[WBF-10]**

ここまでに用意したノードを繋げたワークフローグラフを定義します。

In [107]:
travel_guide_workflow = Workflow(
    name='travel_guide_workflow',
    edges=[
        (
            'START',
            user_information_task, output_user_information,
            travel_guide_agent,
        ),
    ],
)

travel_guide_app = AdkApp(
    agent=travel_guide_workflow,
    app_name='travel_guide_app',
)

**[WBF-11]**

最初のメッセージを入力します。

In [110]:
chat_client = ChatClient(travel_guide_app)

query = '''
こんにちは。
'''
await chat_client.async_stream_query(query, async_output=True)

**[user_information_task]**

こんにちは！はじめまして。
お話しできて嬉しいです♪
さっそくですが、あなたの「お名前」と「ご趣味」を教えていただけますか？

**[WBF-12]**

名前と趣味を聞かれていますが、あえて名前だけを返答します。

In [111]:
query = '''
片桐はいりです。
'''
await chat_client.async_stream_query(query, async_output=True)

**[user_information_task]**

片桐はいりさん、お名前を教えていただきありがとうございます！
とても素敵な名前ですね✨

ちなみに、片桐さんのご趣味は何ですか？ぜひ教えてください！

**[WBF-13]**

趣味も教えるように促されるので、追加で趣味を返答します。

集められた情報がメッセージとして表示されて、さらにこれに基づいて、あいさつの文章が生成されます。

In [112]:
query = '''
映画鑑賞と、遺跡巡りもよく行きます。
'''
await chat_client.async_stream_query(query, async_output=True)

**[travel_guide_workflow]**


```
ユーザー情報を確認しました。
・ 名前: 片桐はいり
・ 趣味: 映画鑑賞、遺跡巡り
```


**[travel_guide_agent]**

映画鑑賞と古代文明のロマンを同時に満たせる映画の聖地、カンボジアのアンコール遺跡群がおすすめです。

**[WBF-14]**

セッション情報に記録された会話履歴を確認します。

In [113]:
session = await chat_client.adk_app.async_get_session(
    user_id = 'default_user',
    session_id = chat_client.session_id,
)

for i, event in enumerate(session.events):
    print(f'\n=== [Event {i+1}] ===')
    print(f'author: {event.author}')
    print(f'content: {event.content}')


=== [Event 1] ===
author: user
content: parts=[Part(
  text="""
こんにちは。
"""
)] role='user'

=== [Event 2] ===
author: user_information_task
content: parts=[Part(
  text="""こんにちは！はじめまして。
お話しできて嬉しいです♪
さっそくですが、あなたの「お名前」と「ご趣味」を教えていただけますか？""",
  thought_signature=b'\x01\x8f=k_`\x9d\xd0\xfe\xc7\xe6\x15|\x06\xc7\x00\x80`\xaf\xb5\x88\xb2\xc7\xb6m\x1b\xad\x14l\xf4z\x83\xa8\x1e(Hc\xe2\x8c\xae\xe0\xe6\xdd\x80\xddH\xcaA\xe7"l\xf7ki\x0b\xc4\xd8$\x08\xb0\xd7#X\x0f<l\x1a\x13:}\xfe\xb7\xa7\xd6\x08\x99\x90\x9c\xe6J\xf3\xb1*'
)] role='model'

=== [Event 3] ===
author: user
content: parts=[Part(
  text="""
片桐はいりです。
"""
)] role='user'

=== [Event 4] ===
author: user_information_task
content: parts=[Part(
  text="""片桐はいりさん、お名前を教えていただきありがとうございます！
とても素敵な名前ですね✨

ちなみに、片桐さんのご趣味は何ですか？ぜひ教えてください！""",
  thought_signature=b"\x01\x8f=k_\xc6\xe3{\x00\x1c\x02\xda\xff\x90\\\x90q\xd2\x173\xbe\xa7 m:t\xee\xec\x88\xd0\xd3Iz\xa9\xe3R\x92\n\x85\xf4~\xe8\x979w\xeap\xe0+\xe9\x1c\xf9\xfb_\xda\xa0~+\xdc[\xa6\x11\x15\xf2'\xb4\xaa

## 分岐処理の実装

**[WBF-15]**

ワークフローを再実行するか確認するタスクノードと、その後処理をする関数ノードを定義します。

In [115]:
class HumanCheckResult(BaseModel):
    """ワークフロー再実行の判断結果"""
    restart: bool = Field(description='判断結果')

instruction = '''
# タスク
1. ワークフローを再実行するかユーザーに質問します。
2. 得られた情報を HumanCheckResult.restart にセットします。
  - 再実行する場合は True
  - 再実行しない場合は False
3. HumanCheckResult.restart をセットしたらタスクを終了します。

# 条件
- 余計な会話はしないで、「はい」か「いいえ」の判断を求めてください。
'''

human_check_task = LlmAgent(
    name='human_check_task',
    model='gemini-3.5-flash-lite',
    mode='task',
    output_schema=HumanCheckResult,
    description='ワークフロー再実行の判断を受け取るタスク',
    instruction=instruction,
)

def process_human_check_result(node_input: HumanCheckResult):
    if node_input.restart:
        return Event(
            message='ワークフローを再実行します。',
            route='restart',
        )
    else:
        return Event(
            message='ワークフローを終了します。',
            route='end',
        )


**[WBF-16]**

分岐処理を追加したワークフローを定義ます。

分岐処理は、関数ノードが出力した Event オブジェクトの `restart` オプションの値で次のノードを決定します。

In [116]:
async def end_node():
    return Event(message='ワークフローが完了しました。')

travel_guide_workflow = Workflow(
    name='travel_guide_workflow',
    edges=[
        (
            'START',
            user_information_task, output_user_information,
            travel_guide_agent,
            human_check_task, process_human_check_result,
        ),
        (
            process_human_check_result,
            {
                'restart': user_information_task,
                DEFAULT_ROUTE: end_node,
            },
        ),
        (
            end_node,
        )
    ],
)

travel_guide_app = AdkApp(
    agent=travel_guide_workflow,
    app_name='travel_guide_app',
)

**[WBF-17]**

最初のメッセージを入力します。

In [117]:
chat_client = ChatClient(travel_guide_app)

query = '''
こんにちは。
'''
await chat_client.async_stream_query(query, async_output=True)

**[user_information_task]**

こんにちは！はじめまして。
楽しくお話ししながら、あなたの「お名前」と「趣味」を教えていただけますか？よろしくお願いします！

**[WBF-18]**

ユーザーの情報を入力します。

In [118]:
query = '''
片桐はいり。映画鑑賞と遺跡巡り。
'''
await chat_client.async_stream_query(query, async_output=True)

**[travel_guide_workflow]**


```
ユーザー情報を確認しました。
・ 名前: 片桐はいり
・ 趣味: 映画鑑賞と遺跡巡り
```


**[travel_guide_agent]**

映画鑑賞と深い歴史ロマンを楽しめる片桐はいりさんには、世界遺産と映画文化が息づくカンボジアの「アンコール・ワット」がおすすめです。

**[human_check_task]**

ワークフローを再実行しますか？「はい」か「いいえ」でお答えください。

**[WBF-19]**

再実行の確認に「はい」で答えます。

In [119]:
query = '''
はい。お願いします。
'''
await chat_client.async_stream_query(query, async_output=True)

**[travel_guide_workflow]**

ワークフローを再実行します。

**[user_information_task]**

こんにちは！はじめまして。
お話しできて嬉しいです♪
よろしければ、あなたの「お名前」と「趣味」を教えていただけますか？

**[WBF-20]**

ユーザーの情報を入力します。

In [120]:
query = '''
小林聡美。俳句と落語が趣味。
'''
await chat_client.async_stream_query(query, async_output=True)

**[travel_guide_workflow]**


```
ユーザー情報を確認しました。
・ 名前: 小林聡美
・ 趣味: 俳句と落語
```


**[travel_guide_agent]**

俳句と落語の趣を味わえる、歴史情緒あふれる東京・浅草の下町散策がおすすめです。

**[human_check_task]**

ワークフローを再実行しますか？「はい」か「いいえ」でお答えください。

**[WBF-21]**

再実行の確認に「中止してください」と答えます。

「はい」「いいえ」以外の表現でも正しく理解されることがわかります。

In [121]:
query = '''
中止してください。
'''
await chat_client.async_stream_query(query, async_output=True)

**[travel_guide_workflow]**

ワークフローを終了します。

**[travel_guide_workflow]**

ワークフローが完了しました。